### Setup Note

This notebook runs on CPU, separate from the DistilBERT training notebook (which is using the Colab GPU). Since this phase is prompt iteration on a small sample of reviews — not the full dataset — CPU is sufficient and avoids competing for GPU memory with the in-progress training run.

In [ ]:

!pip install -q transformers tokenizers
!pip install -q openpyxl

In [ ]:
import transformers
import torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

transformers: 4.51.0
torch: 2.12.1+cu130
GPU: True


In [ ]:

import os, json, shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    DistilBertModel,
    DistilBertTokenizerFast,
    T5ForConditionalGeneration,
    T5Tokenizer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
#from google.colab import files
#print("Upload: train_clean.csv, test_clean.csv, distilbert_10class_best.zip, model_b_best.pt, model_b_vocab.json")
#uploaded = files.upload()

shutil.unpack_archive('distilbert_10class_best.zip', 'distilbert_10class_best')
print("Files ready:", os.listdir('.'))

Files ready: ['.config', 'model_b_vocab.json', 'test_clean.csv', 'distilbert_10class_best', 'model_b_best.pt', 'distilbert_10class_best.zip', 'train_clean.csv', 'sample_data']


In [ ]:
# Load data + rebuild split (same random_state=42 guarantees identical drug/condition mappings as training)
train_df = pd.read_csv('train_clean.csv')
test_df = pd.read_csv('test_clean.csv')

train_split, val_split = train_test_split(
    train_df, test_size=0.15, stratify=train_df['rating'], random_state=42
)
print(f"Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_df)}")

Train: 137102, Val: 24195, Test: 53766


In [ ]:
#  Stratified sample of test set — 1,000 rows, preserving rating distribution
# Using rating as stratification key; bin rare ratings to avoid single-sample strata errors
test_df['rating_bin'] = test_df['rating']  # ratings 1-10 all have enough rows in test set
sample_test = test_df.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(min(len(x), int(np.ceil(1000 * len(x) / len(test_df)))), random_state=42),
    include_groups=False
).head(1000).reset_index(drop=True)

# restore the rating column since include_groups=False drops it from the lambda
sample_test = test_df.groupby('rating', group_keys=False)[test_df.columns].apply(
    lambda x: x.sample(min(len(x), int(np.ceil(1000 * len(x) / len(test_df)))), random_state=42)
).head(1000).reset_index(drop=True)

print(f"Sample size: {len(sample_test)}")
print("Rating distribution in sample:")
print(sample_test['rating'].value_counts(normalize=True).sort_index().round(3))

Sample size: 1000
Rating distribution in sample:
rating
1     0.136
2     0.044
3     0.042
4     0.031
5     0.051
6     0.040
7     0.058
8     0.115
9     0.171
10    0.312
Name: proportion, dtype: float64


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
# Load frozen DistilBERT + tokenizer
text_encoder = DistilBertModel.from_pretrained('distilbert_10class_best', local_files_only=True).to(device)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert_10class_best', local_files_only=True)
text_encoder.eval()
for p in text_encoder.parameters():
    p.requires_grad = False
print("DistilBERT loaded and frozen.")

DistilBERT loaded and frozen.


In [ ]:
# Extract text features for the 1,000-row test sample
MAX_LEN = 160
BATCH_SIZE = 32

encodings = tokenizer(
    sample_test['review'].tolist(),
    truncation=True, padding=True,
    max_length=MAX_LEN, return_tensors='pt'
)

@torch.no_grad()
def extract_features(encodings, batch_size=32):
    feats = []
    for i in range(0, len(encodings['input_ids']), batch_size):
        ids = encodings['input_ids'][i:i+batch_size].to(device)
        mask = encodings['attention_mask'][i:i+batch_size].to(device)
        out = text_encoder(input_ids=ids, attention_mask=mask)
        feats.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(feats, dim=0)

test_text_features = extract_features(encodings)
print(f"Text features shape: {test_text_features.shape}")  # should be [1000, 768]

Text features shape: torch.Size([1000, 768])


### Regenerating Model B's Predictions on the Test Set

Before the generative AI component can run, we need `predicted_rating` for each review in
the sample — Model B's 10-class prediction, using drugName and condition alongside the
review text. This reloads the frozen DistilBERT encoder and Model B's trained head/embeddings
from the checkpoints saved during Chapter 4's training, rather than retraining anything here.

In [ ]:
# Load Model B vocab + architecture + weights
with open('model_b_vocab.json') as f:
    vocab_data = json.load(f)
drug_to_id = vocab_data['drug_to_id']
condition_to_id = vocab_data['condition_to_id']
DRUG_UNK = len(drug_to_id)
CONDITION_UNK = len(condition_to_id)

class ModelB(nn.Module):
    def __init__(self, num_drugs, num_conditions, drug_dim=32, cond_dim=16, num_classes=10):
        super().__init__()
        self.drug_embedding = nn.Embedding(num_drugs + 1, drug_dim)
        self.condition_embedding = nn.Embedding(num_conditions + 1, cond_dim)
        self.classifier = nn.Sequential(
            nn.Linear(768 + drug_dim + cond_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, text_features, drug_ids, condition_ids):
        combined = torch.cat([
            text_features,
            self.drug_embedding(drug_ids),
            self.condition_embedding(condition_ids)
        ], dim=1)
        return self.classifier(combined)

model_b = ModelB(num_drugs=len(drug_to_id), num_conditions=len(condition_to_id)).to(device)
model_b.load_state_dict(torch.load('model_b_best.pt', map_location=device))
model_b.eval()
print("Model B loaded.")

Model B loaded.


In [ ]:
# Generate predicted ratings for the 1,000-row test sample
drug_ids = torch.tensor(
    sample_test['drugName'].apply(lambda x: drug_to_id.get(x, DRUG_UNK)).values
).to(device)
cond_ids = torch.tensor(
    sample_test['condition'].apply(lambda x: condition_to_id.get(x, CONDITION_UNK)).values
).to(device)

with torch.no_grad():
    logits = model_b(test_text_features.to(device), drug_ids, cond_ids)
    predicted_ratings = torch.argmax(logits, dim=1).cpu().numpy() + 1  # convert 0-indexed back to 1-10

sample_test = sample_test.copy()
sample_test['predicted_rating'] = predicted_ratings

# Sanity check
print(f"Predicted rating distribution:")
print(pd.Series(predicted_ratings).value_counts().sort_index())
print(f"\nSample macro-F1: {f1_score(sample_test['rating'], predicted_ratings, average='macro'):.4f}")

Predicted rating distribution:
1     147
2      35
3      46
4      30
5      53
6      43
7      59
8     164
9     183
10    240
Name: count, dtype: int64

Sample macro-F1: 0.5588


In [ ]:
# Add derived columns
def to_3class(rating):
    if rating <= 4:
        return 1
    elif rating < 7:
        return 2
    else:
        return 3

sample_test['rating_class'] = sample_test['rating'].apply(to_3class)
sample_test['predicted_class'] = sample_test['predicted_rating'].apply(to_3class)
sample_test['rating_sentiment'] = sample_test['predicted_rating'].apply(
    lambda x: 'positive' if x > 5 else 'negative'
)

print("Columns so far:", sample_test.columns.tolist())
print("\nClass distribution (true):")
print(sample_test['rating_class'].value_counts().sort_index())
print("\nClass distribution (predicted):")
print(sample_test['predicted_class'].value_counts().sort_index())

Columns so far: ['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'rating_bin', 'predicted_rating', 'rating_class', 'predicted_class', 'rating_sentiment']

Class distribution (true):
rating_class
1    253
2     91
3    656
Name: count, dtype: int64

Class distribution (predicted):
predicted_class
1    258
2     96
3    646
Name: count, dtype: int64


### Loading the Generative Model: Qwen3-4B

The summarization and sentiment task uses **Qwen3-4B** (Alibaba, 2025), a 4-billion parameter
instruction-tuned model, loaded in float16 precision. This size was chosen as a practical
middle ground: large enough to reliably follow the multi-task prompt format (summary +
sentiment in one generation call — a smaller model tested earlier, Flan-T5-base, struggled
with this), while staying small enough to fit comfortably on a free-tier Colab GPU without
quantization.

In [ ]:
# Load Qwen3-4B
!pip install bitsandbytes accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype=torch.float16,
    device_map="auto"
)
qwen_model.eval()
print("Qwen3-4B loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f}GB / 15GB")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B loaded.
GPU memory used: 9.2GB / 15GB


In [ ]:
# Generation helper for Qwen3
def generate_qwen(prompts, max_new_tokens=100):
    results = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        inputs = qwen_tokenizer([text], return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=qwen_tokenizer.eos_token_id
            )
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        result = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        results.append(result)
    return results

### Prompt Engineering: Setting Up the Competition

Before committing to a single prompting strategy and running it at scale, we compare four
approaches on a small, fast sample. A 30-row stratified sample (by rating) keeps the full
four-approach comparison to a few minutes, since the goal here is checking format compliance
and rough sentiment accuracy across rating levels — not statistical significance, which comes
later at the 500-row stage.

In [ ]:
# Stratified sub-sample for prompt competition
# Using only 30 rows to keep experimentation fast (~2-3 min total for 4 approaches)
sub_sample = sample_test.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(min(len(x), max(1, int(np.ceil(30 * len(x) / len(sample_test))))), random_state=42),
    include_groups=True
).head(30).reset_index(drop=True)

print("Prompt competition sub-sample distribution:")
print(sub_sample['rating'].value_counts().sort_index())
print(f"Total: {len(sub_sample)} rows")
print(f"Ratings present: {sorted(sub_sample['rating'].unique())}")

Prompt competition sub-sample distribution:
rating
1     5
2     2
3     2
4     1
5     2
6     2
7     2
8     4
9     6
10    4
Name: count, dtype: int64
Total: 30 rows
Ratings present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


### Four Prompting Approaches

Four prompt designs are defined here, tested against the same 30-row sample:

- **A — Zero-shot structured**: a direct instruction, no examples.
- **B — Simple few-shot**: adds two short, generic examples (one clearly positive, one
  clearly negative) to anchor the expected output format.
- **C — Rich few-shot with real examples**: replaces the generic examples with five real
  reviews from the dataset, spanning ratings 1, 2, 5, 6, and 10.
- **D — Clinically descriptive few-shot**: same five real examples as C, plus role framing
  ("You are a medical data analyst") and an explicit instruction to capture clinical detail
  rather than sentiment alone.

Each function returns a list of formatted prompts, so all four can be run over the same
sample and compared directly.

In [ ]:
# Define 4 prompt approachesn

def make_prompts_a(reviews):
    return [
        f"""Summarize the following drug review in 10 words or less, then classify the sentiment as positive or negative.

Review: {r}

Summary:
Sentiment:""" for r in reviews
    ]

def make_prompts_b(reviews):
    prefix = """Summarize each drug review in 10 words or less, then classify sentiment as positive or negative.

Review: This medication completely changed my life. No side effects at all.
Summary: Life-changing medication with no side effects.
Sentiment: positive

Review: Terrible experience. Made me sick every single day.
Summary: Caused daily sickness, terrible experience overall.
Sentiment: negative

Now do the same:
"""
    return [f"""{prefix}Review: {r}
Summary:
Sentiment:""" for r in reviews]

def make_prompts_c(reviews):
    prefix = """Summarize each drug review in 10 words or less, then classify sentiment as positive or negative.

Review: My daughter began using this drug in December of 2013. On January 25, 2014, she died from a blood clot to her lung. She was not overweight, did not smoke, or had any conditions that were supposed to have negative implications re: use of this drug. This drug, I believe, killed my daughter.
Summary: Drug blamed for daughter's fatal blood clot.
Sentiment: negative

Review: I am only on the beginning of week two, however I've never had a lower sex drive in my life. I've also never been more nauseous in my life. I throw up nightly. Unless it gets better after this packet of pills I'll be calling my pharmacist to switch to a different type of birth control.
Summary: Severe nausea and low libido, considering switching.
Sentiment: negative

Review: I'm currently starting my second pack so I can't give a long term review but I still have some pretty important feedback. Within the second almost third week I started bleeding dark brown blood. What is really nice about this pill is that my body hair has stopped growing. It's not spectacular so far.
Summary: Mixed results; bleeding issues but reduced hair growth.
Sentiment: negative

Review: I've been on this pill for a week and a half. So far the only side effect increased libido, and I mean it's increased a lot. No period, no breast size changes. I will keep taking this pill until I'm ready to settle down and have children.
Summary: Positive experience, only side effect is increased libido.
Sentiment: positive

Review: I am a registered nurse and have had psoriasis since I was 12. I am now 56. 1 month after starting Stelara, I noticed a difference. It has been 9 months and I have no more psoriasis or arthritis pain. I did not experience any side effects. I highly recommend this product.
Summary: Eliminated psoriasis and arthritis pain, highly recommended.
Sentiment: positive

Now do the same:
"""
    return [f"""{prefix}Review: {r}
Summary:
Sentiment:""" for r in reviews]

def make_prompts_d(reviews):
    prefix = """You are a medical data analyst summarizing patient drug reviews. For each review, write a precise summary of exactly 10 words that captures the most specific and clinically meaningful detail from the patient's experience — not just whether they liked the drug, but what actually happened to them. Then classify the overall sentiment as positive or negative.

Review: My daughter began using this drug in December of 2013. On January 25, 2014, she died from a blood clot to her lung. She was not overweight, did not smoke, or had any conditions that were supposed to have negative implications re: use of this drug. This drug, I believe, killed my daughter.
Summary: Daughter died from pulmonary blood clot attributed to drug.
Sentiment: negative

Review: I am only on the beginning of week two, however I've never had a lower sex drive in my life. I've also never been more nauseous in my life. I throw up nightly. Unless it gets better after this packet of pills I'll be calling my pharmacist to switch to a different type of birth control.
Summary: Severe nausea and complete libido loss after two weeks.
Sentiment: negative

Review: I'm currently starting my second pack so I can't give a long term review but I still have some pretty important feedback. Within the second almost third week I started bleeding dark brown blood. What is really nice about this pill is that my body hair has stopped growing. It's not spectacular so far.
Summary: Abnormal bleeding reported but body hair growth significantly reduced.
Sentiment: negative

Review: I've been on this pill for a week and a half. So far the only side effect increased libido, and I mean it's increased a lot. No period, no breast size changes. I will keep taking this pill until I'm ready to settle down and have children.
Summary: Increased libido only side effect; continuing until family planning.
Sentiment: positive

Review: I am a registered nurse and have had psoriasis since I was 12. I am now 56. 1 month after starting Stelara, I noticed a difference. It has been 9 months and I have no more psoriasis or arthritis pain. I did not experience any side effects. I highly recommend this product.
Summary: Nine months of Stelara resolved psoriasis and arthritis completely.
Sentiment: positive

Now do the same — write a precise 10-word summary capturing the specific clinical experience, then classify sentiment:
"""
    return [f"""{prefix}Review: {r}
Summary:
Sentiment:""" for r in reviews]

# Run all 4 on the 100-row sub-sample
print("Running approach A...")
outputs_a = [generate_qwen([p])[0] for p in make_prompts_a(sub_sample['review'].tolist())]
print("Running approach B...")
outputs_b = [generate_qwen([p])[0] for p in make_prompts_b(sub_sample['review'].tolist())]
print("Running approach C...")
outputs_c = [generate_qwen([p])[0] for p in make_prompts_c(sub_sample['review'].tolist())]
print("Running approach D...")
outputs_d = [generate_qwen([p])[0] for p in make_prompts_d(sub_sample['review'].tolist())]
print("All 4 approaches done.")

Running approach A...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Running approach B...
Running approach C...
Running approach D...
All 4 approaches done.


In [ ]:
# pick one example per rating
for target_rating in [1, 5, 10]:
    subset = sub_sample[sub_sample['rating'] == target_rating]
    if len(subset) == 0:
        print(f"\nNo rating-{target_rating} reviews in sub_sample, skipping.")
        continue
    i = sub_sample.index.get_loc(subset.index[0])
    print(f"\n{'='*60}")
    print(f"Rating: {target_rating}")
    print(f"Review: {sub_sample.iloc[i]['review'][:200]}...")
    print(f"\nA: {outputs_a[i]}")
    print(f"\nB: {outputs_b[i]}")
    print(f"\nC: {outputs_c[i]}")
    print(f"\nD: {outputs_d[i]}")


Rating: 1
Review: Taking it for 3 weeks; now am at 2 in am and 2 in pm.   Had little dizziness, which went away.  No other side effects.  Initially it made me hungrier, which I thought was crazy.  This has had no impac...

A: Summary: Drug use for 3 weeks with mild side effects.  
Sentiment: Negative

B: Summary: No significant effects, mild dizziness, no hunger reduction.  
Sentiment: negative

C: Summary: No major side effects but no hunger reduction.  
Sentiment: negative

D: Summary: No effect on hunger or cravings after three weeks.  
Sentiment: negative

Rating: 5
Review: I've had nexplanon since October 2014 and I haven't been a solid 2 weeks without bleeding since. My first period after insertion lasted 6 whole months. Since April I've only had about a week and a hal...

A: Summary: Nexplanon causes irregular bleeding.  
Sentiment: Negative

B: Summary: Nexplanon causes irregular bleeding and long-term use.  
Sentiment: negative

C: Summary: Irregular bleeding with no pregnanc

In [ ]:
# Evaluate sentiment accuracy across all 30 rows
# Ground truth: predicted_rating > 5 = positive, <= 5 = negative
true_sentiment = sub_sample['predicted_rating'].apply(lambda x: 'positive' if x > 5 else 'negative')

def extract_sentiment(output):
    """Extract positive/negative from model output, handling case variations"""
    output_lower = output.lower()
    if 'negative' in output_lower:
        return 'negative'
    elif 'positive' in output_lower:
        return 'positive'
    else:
        return 'unknown'

sent_a = [extract_sentiment(o) for o in outputs_a]
sent_b = [extract_sentiment(o) for o in outputs_b]
sent_c = [extract_sentiment(o) for o in outputs_c]
sent_d = [extract_sentiment(o) for o in outputs_d]

# Calculate accuracy for each approach
for name, preds in [('A', sent_a), ('B', sent_b), ('C', sent_c), ('D', sent_d)]:
    correct = sum(p == t for p, t in zip(preds, true_sentiment))
    unknown = preds.count('unknown')
    print(f"Approach {name}: {correct}/30 correct ({correct/30*100:.1f}%) | Unknown: {unknown}")

# Show breakdown by rating class
print("\nDetailed breakdown by rating:")
results_df = pd.DataFrame({
    'rating': sub_sample['rating'],
    'predicted_rating': sub_sample['predicted_rating'],
    'true_sentiment': true_sentiment,
    'A': sent_a, 'B': sent_b, 'C': sent_c, 'D': sent_d
})
print(results_df[['rating', 'predicted_rating', 'true_sentiment', 'A', 'B', 'C', 'D']].to_string())

Approach A: 25/30 correct (83.3%) | Unknown: 0
Approach B: 24/30 correct (80.0%) | Unknown: 2
Approach C: 23/30 correct (76.7%) | Unknown: 1
Approach D: 24/30 correct (80.0%) | Unknown: 0

Detailed breakdown by rating:
    rating  predicted_rating true_sentiment         A         B         C         D
0        1                 1       negative  negative  negative  negative  negative
1        1                 1       negative  negative  negative  negative  negative
2        1                 3       negative  negative  negative  negative  negative
3        1                 1       negative  negative  negative  negative  negative
4        1                 3       negative  negative  negative  negative  negative
5        2                 2       negative  negative  negative  negative  negative
6        2                 3       negative  negative  negative  negative  negative
7        3                 3       negative  positive  positive  positive  positive
8        3               

### Building the Final Evaluation Sample

Prompt competition is done — B and D are the two approaches moving forward. This builds the
final, larger stratified sample (500 rows, stratified by rating) that both approaches will be
run on. This is a fresh sample, independent of the earlier 1,000-row and 30-row samples used
during setup and prompt competition, drawn with its own random seed (`random_state=99`).

In [ ]:
# Final 500-row stratified sample
final_sample = sample_test.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(
        min(len(x), max(1, int(np.ceil(500 * len(x) / len(sample_test))))),
        random_state=42
    ),
    include_groups=True
).sample(frac=1, random_state=99).head(500).reset_index(drop=True)

print("Final sample distribution:")
print(final_sample['rating'].value_counts().sort_index())
print(f"Total: {len(final_sample)} rows")

Final sample distribution:
rating
1      67
2      22
3      21
4      16
5      25
6      20
7      29
8      58
9      86
10    156
Name: count, dtype: int64
Total: 500 rows


In [ ]:
# Run approach B on final 500-row sample
import warnings
warnings.filterwarnings('ignore')

print("Running approach B on final 500-row sample...")
final_outputs_b = []
for i, review in enumerate(final_sample['review'].tolist()):
    output = generate_qwen(make_prompts_b([review]))[0]
    final_outputs_b.append(output)
    if (i+1) % 50 == 0:
        print(f"  B: {i+1}/500 done")

print("Approach B complete.")

Running approach B on final 500-row sample...
  B: 50/500 done
  B: 100/500 done
  B: 150/500 done
  B: 200/500 done
  B: 250/500 done
  B: 300/500 done
  B: 350/500 done
  B: 400/500 done
  B: 450/500 done
  B: 500/500 done
Approach B complete.


In [ ]:
# Run approach D on final 500-row sample
print("Running approach D on final 500-row sample...")
final_outputs_d = []
for i, review in enumerate(final_sample['review'].tolist()):
    output = generate_qwen(make_prompts_d([review]))[0]
    final_outputs_d.append(output)
    if (i+1) % 50 == 0:
        print(f"  D: {i+1}/500 done")

print("Approach D complete.")

Running approach D on final 500-row sample...
  D: 50/500 done
  D: 100/500 done
  D: 150/500 done
  D: 200/500 done
  D: 250/500 done
  D: 300/500 done
  D: 350/500 done
  D: 400/500 done
  D: 450/500 done
  D: 500/500 done
Approach D complete.


In [ ]:
# Parse outputs — extract summary and sentiment from each raw output string
def extract_summary(output):
    output = output.strip()
    for line in output.split('\n'):
        line = line.strip()
        if line.lower().startswith('summary:'):
            return line[8:].strip()
    return output.split('\n')[0].strip()

def extract_sentiment(output):
    output_lower = output.lower()
    for line in output_lower.split('\n'):
        line = line.strip()
        if line.startswith('sentiment:'):
            val = line[10:].strip()
            if 'negative' in val:
                return 'negative'
            elif 'positive' in val:
                return 'positive'
    if 'negative' in output_lower:
        return 'negative'
    elif 'positive' in output_lower:
        return 'positive'
    return 'unknown'

# Parse B outputs
summaries_b = [extract_summary(o) for o in final_outputs_b]
sentiments_b = [extract_sentiment(o) for o in final_outputs_b]

# Parse D outputs
summaries_d = [extract_summary(o) for o in final_outputs_d]
sentiments_d = [extract_sentiment(o) for o in final_outputs_d]

# Quick sanity check
print(f"B — unknowns: {sentiments_b.count('unknown')}/500")
print(f"D — unknowns: {sentiments_d.count('unknown')}/500")
print(f"\nSample B outputs:")
for i in range(3):
    print(f"  Summary: {summaries_b[i]}")
    print(f"  Sentiment: {sentiments_b[i]}")
    print()

B — unknowns: 57/500
D — unknowns: 8/500

Sample B outputs:
  Summary: Long-term effective medication with no side effects.
  Sentiment: positive

  Summary: Veramyst reduced fluid, improved oxygen levels, no side effects.
  Sentiment: positive

  Summary: Contrave caused headaches, thirst, and constipation initially.
  Sentiment: unknown



In [ ]:
# Sentiment accuracy vs rating_sentiment
true_sent = final_sample['rating_sentiment'].tolist()

correct_b = sum(p == t for p, t in zip(sentiments_b, true_sent))
correct_d = sum(p == t for p, t in zip(sentiments_d, true_sent))

print(f"Approach B sentiment accuracy: {correct_b}/500 ({correct_b/500*100:.1f}%)")
print(f"Approach D sentiment accuracy: {correct_d}/500 ({correct_d/500*100:.1f}%)")

# Agreement between B and D
agree = sum(b == d for b, d in zip(sentiments_b, sentiments_d))
print(f"\nAgreement between B and D: {agree}/500 ({agree/500*100:.1f}%)")

Approach B sentiment accuracy: 388/500 (77.6%)
Approach D sentiment accuracy: 386/500 (77.2%)

Agreement between B and D: 416/500 (83.2%)


In [ ]:
from sklearn.metrics import classification_report

print("Approach B:")
print(classification_report(true_sent, sentiments_b, labels=['positive', 'negative']))

print("Approach D:")
print(classification_report(true_sent, sentiments_d, labels=['positive', 'negative']))

Approach B:
              precision    recall  f1-score   support

    positive       0.97      0.72      0.83       345
    negative       0.75      0.90      0.82       155

   micro avg       0.88      0.78      0.82       500
   macro avg       0.86      0.81      0.82       500
weighted avg       0.90      0.78      0.82       500

Approach D:
              precision    recall  f1-score   support

    positive       0.97      0.69      0.81       345
    negative       0.60      0.95      0.73       155

   micro avg       0.78      0.77      0.78       500
   macro avg       0.78      0.82      0.77       500
weighted avg       0.85      0.77      0.78       500



In [ ]:
# Check unknown distribution by rating class
final_sample['sentiment_b'] = sentiments_b
final_sample['sentiment_d'] = sentiments_d

print("Approach B — unknowns by rating class:")
b_unknowns = final_sample[final_sample['sentiment_b'] == 'unknown']
print(b_unknowns['rating_class'].value_counts().sort_index())
print(f"Total B unknowns: {len(b_unknowns)}")

print("\nApproach D — unknowns by rating class:")
d_unknowns = final_sample[final_sample['sentiment_d'] == 'unknown']
print(d_unknowns['rating_class'].value_counts().sort_index())
print(f"Total D unknowns: {len(d_unknowns)}")

print("\nFor reference — full sample class distribution:")
print(final_sample['rating_class'].value_counts().sort_index())

Approach B — unknowns by rating class:
rating_class
1     5
2     7
3    45
Name: count, dtype: int64
Total B unknowns: 57

Approach D — unknowns by rating class:
rating_class
2    1
3    7
Name: count, dtype: int64
Total D unknowns: 8

For reference — full sample class distribution:
rating_class
1    126
2     45
3    329
Name: count, dtype: int64


### Building the Two Output Excel Files

This produces the two deliverable files, one for each of the selected prompting approaches
(B and D), both built with the same schema: `drugName`, `condition`, `review`, `rating`,
`predicted_rating`, `rating_class`, `predicted_class`, `generated_summary`,
`identified_sentiment`, and `rating_sentiment` (the rating-derived sentiment, kept as a
cross-reference against the model-generated `identified_sentiment` — see the discussion in
the report on where the two agree and disagree).

In [ ]:
# Build Excel files
import openpyxl

def build_excel(df, summaries, sentiments, filename):
    output_df = pd.DataFrame({
        'drugName': df['drugName'],
        'condition': df['condition'],
        'review': df['review'],
        'rating': df['rating'],
        'predicted_rating': df['predicted_rating'],
        'rating_class': df['rating_class'],
        'predicted_class': df['predicted_class'],
        'generated_summary': summaries,
        'identified_sentiment': sentiments,
        'rating_sentiment': df['rating_sentiment']
    })
    output_df.to_excel(filename, index=False)
    print(f"Saved: {filename} ({len(output_df)} rows)")
    return output_df

df_b = build_excel(final_sample, summaries_b, sentiments_b, 'output_approach_b.xlsx')
df_d = build_excel(final_sample, summaries_d, sentiments_d, 'output_approach_d.xlsx')

Saved: output_approach_b.xlsx (500 rows)
Saved: output_approach_d.xlsx (500 rows)


In [ ]:
# Load Excel files and analyze unknowns
import pandas as pd

df_b = pd.read_excel('output_approach_b.xlsx')
df_d = pd.read_excel('output_approach_d.xlsx')

print(f"B shape: {df_b.shape}, D shape: {df_d.shape}")
print(f"\nB unknowns: {(df_b['identified_sentiment'] == 'unknown').sum()}")
print(f"D unknowns: {(df_d['identified_sentiment'] == 'unknown').sum()}")

B shape: (500, 10), D shape: (500, 10)

B unknowns: 57
D unknowns: 8


In [ ]:
# Find cases where only B is unknown (D has an answer)
b_unknown_mask = df_b['identified_sentiment'] == 'unknown'
d_unknown_mask = df_d['identified_sentiment'] == 'unknown'

# Case 1: Only B unknown, D has answer (disagreement)
only_b_unknown = df_b[b_unknown_mask & ~d_unknown_mask].index
print(f"Only B unknown (D has answer): {len(only_b_unknown)} cases")

# Case 2: Both unknown
both_unknown = df_b[b_unknown_mask & d_unknown_mask].index
print(f"Both unknown: {len(both_unknown)} cases")

# Show one example from each case
cols = ['review', 'rating', 'generated_summary', 'identified_sentiment']

print("\n=== CASE 1: Only B unknown, D has answer ===")
idx = only_b_unknown[0]
print(f"Rating: {df_b.loc[idx, 'rating']}")
print(f"Review: {df_b.loc[idx, 'review'][:300]}")
print(f"B summary: {df_b.loc[idx, 'generated_summary']}")
print(f"B sentiment: {df_b.loc[idx, 'identified_sentiment']}")
print(f"D summary: {df_d.loc[idx, 'generated_summary']}")
print(f"D sentiment: {df_d.loc[idx, 'identified_sentiment']}")

print("\n=== CASE 2: Both B and D unknown ===")
idx2 = both_unknown[0]
print(f"Rating: {df_b.loc[idx2, 'rating']}")
print(f"Review: {df_b.loc[idx2, 'review'][:300]}")
print(f"B summary: {df_b.loc[idx2, 'generated_summary']}")
print(f"B sentiment: {df_b.loc[idx2, 'identified_sentiment']}")
print(f"D summary: {df_d.loc[idx2, 'generated_summary']}")
print(f"D sentiment: {df_d.loc[idx2, 'identified_sentiment']}")

Only B unknown (D has answer): 53 cases
Both unknown: 4 cases

=== CASE 1: Only B unknown, D has answer ===
Rating: 8
Review: I am on day 15 of taking Contrave. I took one pill/day for the first 10 days & have been on 2/day for these last 5. Days 1-6: gnawing headache at least once/day; insatiable thirst; significant constipation. During that time, I had decreased appetite, but I think that may have been due to the constip
B summary: Contrave caused headaches, thirst, and constipation initially.
B sentiment: unknown
D summary: Headache, thirst, constipation resolved with miralax and increased dose.
D sentiment: positive

=== CASE 2: Both B and D unknown ===
Rating: 9
Review: Chantix is an amazing medicine for smoking cessation. It completely robs you of any pleasant sensations you got from smoking. That said, I have experienced a lot of the side effects including severe nausea, headaches, extremely vivid but good dreams (when have you ever been able to feel the texture 
B summary: Effe

In [ ]:
# Check summary word counts for both approaches
def count_words(summary):
    return len(str(summary).split())

df_b['summary_word_count'] = df_b['generated_summary'].apply(count_words)
df_d['summary_word_count'] = df_d['generated_summary'].apply(count_words)

# Percentage exceeding 10 words
b_over_10 = (df_b['summary_word_count'] > 10).sum()
d_over_10 = (df_d['summary_word_count'] > 10).sum()

print(f"Approach B — summaries exceeding 10 words: {b_over_10}/500 ({b_over_10/500*100:.1f}%)")
print(f"Approach D — summaries exceeding 10 words: {d_over_10}/500 ({d_over_10/500*100:.1f}%)")

print(f"\nWord count distribution — Approach B:")
print(df_b['summary_word_count'].value_counts().sort_index())

print(f"\nWord count distribution — Approach D:")
print(df_d['summary_word_count'].value_counts().sort_index())

Approach B — summaries exceeding 10 words: 13/500 (2.6%)
Approach D — summaries exceeding 10 words: 67/500 (13.4%)

Word count distribution — Approach B:
summary_word_count
2       2
3       3
4       9
5      41
6      90
7     122
8     118
9      73
10     29
11     11
12      2
Name: count, dtype: int64

Word count distribution — Approach D:
summary_word_count
2       1
5      13
6      29
7      72
8     136
9     112
10     70
11     42
12     22
13      3
Name: count, dtype: int64


### Model B's Rating-Prediction Performance on the Held-Out Test Sample

Everything up to this point in the notebook has been about the generative summarization
and sentiment task. This last check circles back to the *original* rating-prediction task:
now that `predicted_rating` (Model B's 10-class prediction) has been generated for this
500-row sample, we can evaluate it the same way Models A and B were evaluated on the
validation set in Chapter 4 — accuracy, macro-F1, and ±1 accuracy for the 10-class task,
plus accuracy and macro-F1 for the re-bucketed 3-class version.

This is the first time Model B is evaluated on genuinely held-out **test** data (rather than
the validation split it was trained and tuned against), and on a much smaller sample (500
rows vs. the full 24,195-row validation set) — so these numbers are reported as a sanity
check on generalization, not as a replacement for the validation-set results.

In [ ]:
from sklearn.metrics import classification_report, f1_score

# 10-class performance on the 500-row sample
print("=== 10-Class Performance (500-row test sample) ===")
print(f"Accuracy: {(df_b['rating'] == df_b['predicted_rating']).mean():.4f}")
print(f"Macro-F1: {f1_score(df_b['rating'], df_b['predicted_rating'], average='macro'):.4f}")
print(f"±1 accuracy: {(abs(df_b['rating'] - df_b['predicted_rating']) <= 1).mean():.4f}")

# 3-class performance on the 500-row sample
print("\n=== 3-Class Performance (500-row test sample) ===")
print(f"Accuracy: {(df_b['rating_class'] == df_b['predicted_class']).mean():.4f}")
print(f"Macro-F1: {f1_score(df_b['rating_class'], df_b['predicted_class'], average='macro'):.4f}")
print(classification_report(df_b['rating_class'], df_b['predicted_class'],
                           target_names=['Class 1 (≤4)', 'Class 2 (4-7)', 'Class 3 (≥7)']))

=== 10-Class Performance (500-row test sample) ===
Accuracy: 0.5800
Macro-F1: 0.5761
±1 accuracy: 0.8360

=== 3-Class Performance (500-row test sample) ===
Accuracy: 0.9080
Macro-F1: 0.8378
               precision    recall  f1-score   support

 Class 1 (≤4)       0.89      0.91      0.90       126
Class 2 (4-7)       0.65      0.69      0.67        45
 Class 3 (≥7)       0.95      0.94      0.94       329

     accuracy                           0.91       500
    macro avg       0.83      0.85      0.84       500
 weighted avg       0.91      0.91      0.91       500

